In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **128-Channel ERP-EEG Dataset (.raw Files)**

In [ ]:
import mne
import os

# Define a function to preprocess a single .raw file
def preprocess_raw_file(file_path):
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True)
    except ValueError:
        raw = load_custom_raw(file_path)  # Use your custom loader if needed

    # Apply preprocessing steps
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    raw.set_montage(montage)
    raw.filter(l_freq=1., h_freq=30.)
    raw.notch_filter(freqs=50)
    ica = mne.preprocessing.ICA(n_components=20, random_state=97)
    ica.fit(raw)
    ica.exclude = [0]  # Adjust based on analysis
    raw = ica.apply(raw)
    raw.resample(sfreq=250)

    return raw
def preprocess_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.raw'):
            file_path = os.path.join(input_dir, file_name)
            try:
                raw = preprocess_raw_file(file_path)
            except Exception as e:
                print(f"Error processing {file_name}: {e}")
                continue

            # Save the preprocessed file
            output_path = os.path.join(output_dir, file_name.replace('.raw', 'raw.fif'))
            raw.save(output_path, overwrite=True)
            print(f"Processed and saved: {file_name}")
def visualize_electrode_placement():
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    montage.plot()
def compute_and_visualize_psd(raw_data):
    raw_data.plot_psd(fmin=1, fmax=30, average=True, spatial_colors=True)
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data'

# Preprocess the directory of .raw files
preprocess_directory(input_dir, output_dir)

# Visualize electrode placement
visualize_electrode_placement()

# Optionally, compute and visualize PSD of preprocessed data
# Example assuming you've loaded a preprocessed file
raw_data = mne.io.read_raw_fif('/kaggle/working/processed_raw_data/', preload=True)
compute_and_visualize_psd(raw_data)


In [ ]:
import mne
import numpy as np
import os

# Function to load custom .raw file
def load_custom_raw(file_path):
    # Implement your custom loader logic here
    data = np.loadtxt(file_path)  # Example, replace with actual reading logic
    
    # Create info object
    ch_names = ['EEG {}'.format(i) for i in range(data.shape[0])]  # Adjust based on your channel names
    ch_types = ['eeg'] * data.shape[0]  # Assuming all channels are EEG, adjust if needed
    sfreq = 250  # Adjust based on your sampling frequency
    
    info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)
    
    # Create RawArray object
    raw = mne.io.RawArray(data, info)
    
    return raw

# Function to preprocess a single .raw file
def preprocess_raw_file(file_path):
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True)
    except ValueError:
        raw = load_custom_raw(file_path)  # Use your custom loader if needed

    # Apply preprocessing steps
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    raw.set_montage(montage)
    raw.filter(l_freq=1., h_freq=30.)
    raw.notch_filter(freqs=50)
    ica = mne.preprocessing.ICA(n_components=20, random_state=97)
    ica.fit(raw)
    ica.exclude = [0]  # Adjust based on analysis
    raw = ica.apply(raw)
    raw.resample(sfreq=250)

    return raw

# Function to preprocess a directory of .raw files
def preprocess_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.raw'):
            file_path = os.path.join(input_dir, file_name)
            try:
                raw = preprocess_raw_file(file_path)
            except Exception as e:
                print(f"Error processing {file_name}: {e}")
                continue

            # Save the preprocessed file
            output_path = os.path.join(output_dir, file_name.replace('.raw', '_preprocessed.fif'))
            raw.save(output_path, overwrite=True)
            print(f"Processed and saved: {file_name}")

# Function to visualize electrode placement
def visualize_electrode_placement():
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    montage.plot()

# Function to compute and visualize Power Spectral Density (PSD)
def compute_and_visualize_psd(raw_data):
    raw_data.plot_psd(fmin=1, fmax=30, average=True, spatial_colors=True)

# Example usage
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data'

# Preprocess the directory of .raw files
preprocess_directory(input_dir, output_dir)

# Visualize electrode placement
visualize_electrode_placement()

# Optionally, compute and visualize PSD of preprocessed data
# Example assuming you've loaded a preprocessed file
# raw_data = mne.io.read_raw_fif('/path/to/preprocessed_file.fif', preload=True)
# compute_and_visualize_psd(raw_data)


In [ ]:
import mne
import numpy as np
import os

# Function to load custom .raw file
def load_custom_raw(file_path):
    # Implement your custom loader logic here
    data = np.loadtxt(file_path)  # Example, replace with actual reading logic
    
    # Create info object
    ch_names = ['EEG {}'.format(i) for i in range(data.shape[0])]  # Adjust based on your channel names
    ch_types = ['eeg'] * data.shape[0]  # Assuming all channels are EEG, adjust if needed
    sfreq = 250  # Adjust based on your sampling frequency
    
    info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)
    
    # Create RawArray object
    raw = mne.io.RawArray(data, info)
    
    return raw

# Function to preprocess a single .raw file
def preprocess_raw_file(file_path):
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True)
    except ValueError:
        raw = load_custom_raw(file_path)  # Use your custom loader if needed

    # Apply preprocessing steps
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    raw.set_montage(montage)
    raw.filter(l_freq=1., h_freq=30.)
    raw.notch_filter(freqs=50)
    ica = mne.preprocessing.ICA(n_components=20, random_state=97)
    ica.fit(raw)
    ica.exclude = [0]  # Adjust based on analysis
    raw = ica.apply(raw)
    raw.resample(sfreq=250)

    return raw

# Function to preprocess a directory of .raw files
def preprocess_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.raw'):
            file_path = os.path.join(input_dir, file_name)
            try:
                raw = preprocess_raw_file(file_path)
            except Exception as e:
                print(f"Error processing {file_name}: {e}")
                continue

            # Save the preprocessed file
            output_path = os.path.join(output_dir, file_name.replace('.raw', '_preprocessed.fif'))
            raw.save(output_path, overwrite=True)
            print(f"Processed and saved: {file_name}")

# Function to visualize electrode placement
def visualize_electrode_placement():
    montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
    montage.plot()

# Function to compute and visualize Power Spectral Density (PSD)
def compute_and_visualize_psd(raw_data):
    raw_data.plot_psd(fmin=1, fmax=30, average=True, spatial_colors=True)

# Example usage
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/processed_raw_data'

# Preprocess the directory of .raw files
preprocess_directory(input_dir, output_dir)

# Visualize electrode placement
visualize_electrode_placement()

# Optionally, compute and visualize PSD of preprocessed data
# Example assuming you've loaded a preprocessed file
# raw_data = mne.io.read_raw_fif('/path/to/preprocessed_file.fif', preload=True)
# compute_and_visualize_psd(raw_data)


In [ ]:
import mne
import os

def preprocess_raw_file(file_path):
    try:
        # Load the raw file
        raw = mne.io.read_raw_egi(file_path, preload=True)
        print(f"Loaded {file_path}")

        # Check for stimulus channel
        if 'STI 014' not in raw.ch_names:
            print(f"Missing 'STI 014' channel in {file_path}. Skipping.")
            return None

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        montage.plot(kind='3d')
        montage.plot(kind='topomap', show_names=False)

        return raw

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .raw files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_raw_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .raw files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.raw'):
        file_path = os.path.join(input_dir, file_name)
        raw = preprocess_raw_file(file_path)
        if raw:
            # Save the preprocessed file
            output_file_name = file_name.replace('.raw', '-epo.fif')
            output_path = os.path.join(output_dir, output_file_name)
            raw.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import mne
import os
import matplotlib.pyplot as plt

def preprocess_raw_file(file_path, output_dir):
    try:
        # Load the raw file
        raw = mne.io.read_raw_egi(file_path, preload=True)
        print(f"Loaded {file_path}")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
        
        # Check for missing channels and rename if necessary
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)
        
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")
        
        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d')
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Save the preprocessed file
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-preprocessed.fif'
        output_path = os.path.join(output_dir, output_file_name)
        raw.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return raw

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .raw files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_ERP_lanzhou_2015/EEG_128channels_ERP_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_raw_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .raw files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.raw'):
        file_path = os.path.join(input_dir, file_name)
        raw = preprocess_raw_file(file_path, output_dir)
        if raw:
            # Save the preprocessed file
            output_file_name = file_name.replace('.raw', '-preprocessed.fif')
            output_path = os.path.join(output_dir, output_file_name)
            raw.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


# **128-Channel ERP-EEG Dataset (.mat Files)**

In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data and sampling frequency
        data_key = None

        for key in keys:
            if 'rest' in key:
                data_key = key

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
        
        # Check for missing channels and rename if necessary
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)
        
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")
        
        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d')
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print("Epochs created.")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            output_path = os.path.join(output_dir, file_name.replace('.mat', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'rest' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Check data shape
        print(f"Data shape: {data.shape}")

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Check for missing channels and rename if necessary
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)

        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d')
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        # Save the epochs to a .fif file
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-epo.fif'
        output_path = os.path.join(output_dir, output_file_name)
        epochs.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            output_path = os.path.join(output_dir, file_name.replace('.mat', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt

def adaptive_noise_cancellation(raw, lms_lr=0.01, n_iter=1):
    """Adaptive noise cancellation using LMS algorithm."""
    data = raw.get_data()
    n_channels, n_times = data.shape
    reference_channel = data[-1, :]  # Assuming the last channel is used as reference

    for _ in range(n_iter):
        for i in range(n_channels - 1):  # Exclude the reference channel
            # Apply LMS algorithm
            filter_weights = np.zeros(n_times)
            filtered_output = np.zeros(n_times)
            for t in range(1, n_times):
                prediction = filter_weights[t - 1] * reference_channel[t - 1]
                error = data[i, t] - prediction
                filter_weights[t] = filter_weights[t - 1] + lms_lr * error * reference_channel[t - 1]
                filtered_output[t] = error

            data[i, :] = filtered_output

    return mne.io.RawArray(data, raw.info)

def preprocess_mat_file(file_path):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'rest' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Check data shape
        print(f"Data shape: {data.shape}")

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Handle missing channels
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)

        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d', show_names=True)
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Apply LMS-based adaptive filter for blink artifact removal
        raw = adaptive_noise_cancellation(raw)
        print("Adaptive noise cancellation applied.")

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path)
        if epochs:
            output_path = os.path.join(output_dir, file_name.replace('.mat', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import signal

def adaptive_noise_cancellation_lms(raw, reference_idx, mu=0.01, n_iter=1):
    """Apply LMS adaptive noise cancellation."""
    data = raw.get_data()
    ref = data[reference_idx, :]
    
    for _ in range(n_iter):
        for i in range(data.shape[0]):
            if i != reference_idx:
                error = data[i, :] - ref
                weights = mu * error * ref
                data[i, :] = data[i, :] - weights
                
    raw._data = data
    return raw

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'rest' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Apply LMS-based blink artifact removal
        raw = adaptive_noise_cancellation_lms(raw, reference_idx=0)  # Assuming channel 0 is reference; adjust as needed
        print("LMS-based artifact removal applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Handle missing channels
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            raw.info['bads'] += missing_chs
            raw.drop_channels(missing_chs)

        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        fig = raw.plot_sensors(kind='3d', show_names=True)
        plt.show()  # Show the plot
        fig2 = raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()  # Show the plot

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        # Save the epochs
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-epo.fif'
        output_path = os.path.join(output_dir, output_file_name)
        epochs.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            output_path = os.path.join(output_dir, file_name.replace('.mat', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import signal

def adaptive_noise_cancellation_lms(raw, reference_idx, mu=0.01, n_iter=1):
    """Apply LMS adaptive noise cancellation."""
    data = raw.get_data()
    ref = data[reference_idx, :]
    
    for _ in range(n_iter):
        for i in range(data.shape[0]):
            if i != reference_idx:
                error = data[i, :] - ref
                weights = mu * error * ref
                data[i, :] = data[i, :] - weights
                
    raw._data = data
    return raw

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'rest' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Apply LMS-based blink artifact removal
        raw = adaptive_noise_cancellation_lms(raw, reference_idx=0)  # Adjust reference index as needed
        print("LMS-based artifact removal applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Match and rename channels if needed
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            # Rename channels if needed
            rename_dict = {ch: montage.ch_names[i] for i, ch in enumerate(raw.ch_names) if ch not in montage.ch_names}
            raw.rename_channels(rename_dict)
            # Drop channels still missing after renaming
            missing_chs_after_rename = [ch for ch in raw.ch_names if ch not in montage.ch_names]
            raw.drop_channels(missing_chs_after_rename)
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        raw.plot_sensors(kind='3d', show_names=True)
        plt.show()
        raw.plot_sensors(kind='topomap', show_names=False)
        plt.show()

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        # Save the epochs
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-epo.fif'
        output_path = os.path.join(output_dir, output_file_name)
        epochs.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            output_path = os.path.join(output_dir, file_name.replace('.mat', '-epo.fif'))
            epochs.save(output_path, overwrite=True)
            print(f"Saved preprocessed data to {output_path}")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
import scipy.io
import mne
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import signal

def adaptive_noise_cancellation_lms(raw, reference_idx, mu=0.01, n_iter=1):
    """Apply LMS adaptive noise cancellation."""
    data = raw.get_data()
    ref = data[reference_idx, :]
    
    for _ in range(n_iter):
        for i in range(data.shape[0]):
            if i != reference_idx:
                error = data[i, :] - ref
                weights = mu * error * ref
                data[i, :] = data[i, :] - weights
                
    raw._data = data
    return raw

def preprocess_mat_file(file_path, output_dir):
    try:
        # Load .mat file
        mat = scipy.io.loadmat(file_path)
        print(f"Loaded {file_path}")

        # Inspect the structure
        keys = list(mat.keys())
        print("Keys in .mat file:", keys)

        # Find the correct key for the EEG data
        data_key = None
        for key in keys:
            if 'Impedances_0' in key:
                data_key = key
                break

        if data_key is None:
            raise KeyError("EEG data key not found in the .mat file.")

        # Extract data
        data = mat[data_key]

        # Check data dimensions and transpose if necessary
        if data.shape[0] > data.shape[1]:
            data = data.T

        # Create MNE info structure
        sfreq = 250  # Sampling frequency set to 250 Hz
        ch_names = [f'EEG {i+1:03d}' for i in range(data.shape[0])]
        info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

        # Create RawArray object
        raw = mne.io.RawArray(data, info)
        print("Raw data created.")

        # Apply bandpass filtering
        raw.filter(l_freq=1.0, h_freq=30.0)
        print("Bandpass filtering (1.0 - 30.0 Hz) applied.")

        # Apply LMS-based blink artifact removal
        raw = adaptive_noise_cancellation_lms(raw, reference_idx=0)  # Adjust reference index as needed
        print("LMS-based artifact removal applied.")

        # Set EEG reference to average
        raw.set_eeg_reference('average')
        print("Re-referencing to average applied.")

        # Apply standard montage
        montage = mne.channels.make_standard_montage('GSN-HydroCel-128')

        # Match and rename channels if needed
        missing_chs = [ch for ch in raw.ch_names if ch not in montage.ch_names]
        if missing_chs:
            print(f"Missing channels in montage: {missing_chs}")
            # Rename channels if needed
            rename_dict = {ch: montage.ch_names[i] for i, ch in enumerate(raw.ch_names) if ch not in montage.ch_names}
            raw.rename_channels(rename_dict)
            # Drop channels still missing after renaming
            missing_chs_after_rename = [ch for ch in raw.ch_names if ch not in montage.ch_names]
            raw.drop_channels(missing_chs_after_rename)
        raw.set_montage(montage)
        print("Standard montage 'GSN-HydroCel-128' applied.")

        # Plot electrode placement in 2D and 3D
        fig1 = raw.plot_sensors(kind='3d', show_names=True, show=False)
        fig2 = raw.plot_sensors(kind='topomap', show_names=False, show=False)
        
        # Save the plots
        plot_path_3d = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(file_path))[0]}_3d.png")
        plot_path_topomap = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(file_path))[0]}_topomap.png")
        fig1.savefig(plot_path_3d)
        fig2.savefig(plot_path_topomap)
        plt.close(fig1)
        plt.close(fig2)

        # Define events manually if needed or use a placeholder
        events = np.array([[i, 0, 1] for i in range(0, raw.n_times, int(sfreq))])
        event_id = {'Stimulus': 1}  # Placeholder event ID

        # Epoching
        epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=-0.2, tmax=0.8, preload=True)
        print(f"Epochs created with {len(epochs)} epochs.")

        # Save the epochs
        output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-epo.fif'
        output_path = os.path.join(output_dir, output_file_name)
        epochs.save(output_path, overwrite=True)
        print(f"Saved preprocessed data to {output_path}")

        return epochs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Directory containing .mat files
input_dir = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015'
output_dir = '/kaggle/working/preprocessed_mat_output_directory'
os.makedirs(output_dir, exist_ok=True)

# Process all .mat files in the directory
for file_name in os.listdir(input_dir):
    if file_name.endswith('.mat'):
        file_path = os.path.join(input_dir, file_name)
        epochs = preprocess_mat_file(file_path, output_dir)
        if epochs:
            print(f"Preprocessing complete for {file_name}.")
        else:
            print(f"Skipping {file_name} due to errors.")


In [ ]:
epochs._get_data()

In [ ]:
epochs.plot_drop_log()

In [ ]:
import scipy.io as sio

file_path = '/kaggle/input/modma-dataset/EEG_128channels_resting_lanzhou_2015/EEG_128channels_resting_lanzhou_2015/02010005rest 20150507 0907..mat'  # Update this path
mat_contents = sio.loadmat(file_path)
print(mat_contents.keys())  # Print all keys in the MAT file
